In [2]:
# --- Setup: imports ---
import os
import numpy as np
import pandas as pd

# --- Funciones utilitarias ---

IDENTITY_COLS_DEFAULT = [
    # género
    "male", "female", "transgender", "other_gender",
    # orientación sexual
    "heterosexual", "gay", "lesbian", "bisexual", "other_sexual_orientation",
    # religión
    "christian", "jewish", "muslim", "hindu", "buddhist",
    "atheist", "other_religion",
    # raza/etnia
    "black", "white", "asian", "latino", "other_race_or_ethnicity",
    # otros
    "physical_disability", "intellectual_or_learning_disability",
    "psychiatric_or_mental_illness",
]

def ensure_identity_any(df: pd.DataFrame, force_recompute=False) -> pd.DataFrame:
    """
    Crea 'identity_any' si no existe (o si force_recompute=True),
    como OR (>=0.5) sobre las columnas de identidad disponibles.
    """
    identity_cols = [c for c in IDENTITY_COLS_DEFAULT if c in df.columns]

    if force_recompute or ("identity_any" not in df.columns):
        if not identity_cols:
            raise ValueError(
                "No encontré columnas de identidad. "
                "Agrega 'identity_any' al CSV o ajusta IDENTITY_COLS_DEFAULT."
            )
        df["identity_any"] = (df[identity_cols] >= 0.5).any(axis=1).astype(float)
    return df

def make_groups(df: pd.DataFrame,
                target_col="toxicity",
                confounder_cols=("identity_any",)):

    # y binaria a partir de target_col
    y = (df[target_col].values >= 0.5).astype(int)
    n_classes = 2

    # confounders binarios (umbral 0.5) -> código binario
    for c in confounder_cols:
        if c not in df.columns:
            raise ValueError(f"Falta columna de confounder: {c}")
    conf_bin = (df[list(confounder_cols)].values >= 0.5).astype(int)
    conf_code = (conf_bin * (2 ** np.arange(conf_bin.shape[1]))).sum(axis=1)

    # mapeo a grupos (igual que tu JigsawDataset)
    n_groups = n_classes * (2 ** len(confounder_cols))
    groups = (y * (n_groups // n_classes) + conf_code).astype(int)
    return y, groups, n_groups

def balanced_sample(group_array, per_group=None, seed=0):
    """
    Índices de un muestreo balanceado por grupo.
    - per_group=None: usa el mínimo conteo (balance máximo).
    - per_group=int: toma min(per_group, conteo_grupo).
    """
    rng = np.random.default_rng(seed)
    idx = np.arange(len(group_array))
    unique_groups, counts = np.unique(group_array, return_counts=True)

    if per_group is None:
        k = counts.min()
    else:
        k = int(per_group)

    chosen = []
    for g in unique_groups:
        g_idx = idx[group_array == g]
        take = min(len(g_idx), k)
        chosen.extend(rng.choice(g_idx, size=take, replace=False))
    return np.array(sorted(chosen))

def show_group_counts(name, groups):
    vals, cnts = np.unique(groups, return_counts=True)
    print(f"{name} - grupos y conteos:")
    for v, c in zip(vals, cnts):
        print(f"  g={v}: {c}")


In [3]:
# --- Parámetros ---
CSV_PATH  = "../datasets/jigsaw/data/all_data_with_identities.csv"   # <-- ajusta
OUT_DIR   = "../datasets/jigsaw/data/dfr"                 # carpeta de salida

TARGET_COL = "toxicity"
CONFOUNDERS = ("identity_any",)  # o múltiples: ("black","christian") etc.

PER_GROUP = None   # None => balance máximo (mínimo conteo entre grupos)
SEED      = 0
FORCE_RECOMPUTE_ID_ANY = False
WRITE_METADATA_VAL_AS_TRAIN = True  # crea un CSV donde ese val balanceado pasa a 'train'


In [5]:
# --- Cargar CSV ---
df = pd.read_csv(CSV_PATH)

# Si viene con columna índice "Unnamed: 0", úsala como índice (opcional)
if "Unnamed: 0" in df.columns and df.index.name is None:
    df = df.set_index("Unnamed: 0")

# Asegurar texto bien tipado (si luego reutilizas con el Dataset)
if "comment_text" in df.columns:
    df["comment_text"] = df["comment_text"].fillna("").astype(str)

# Asegurar 'identity_any' si hace falta
df = ensure_identity_any(df, force_recompute=FORCE_RECOMPUTE_ID_ANY)

# Filtrar validación (CivilComments suele usar 'train'/'val'/'test' como strings)
if "split" not in df.columns:
    raise ValueError("El CSV no tiene columna 'split' (train/val/test).")

val_mask = df["split"] == "val"
val_df = df[val_mask].copy()
if val_df.empty:
    raise ValueError("No hay filas con split == 'val'.")

# y, grupos en validación
y_val, g_val, n_groups = make_groups(val_df, target_col=TARGET_COL, confounder_cols=CONFOUNDERS)

print(f"n_groups = {n_groups}")
show_group_counts("Validación (antes)", g_val)

# muestreo balanceado
chosen_idx_local  = balanced_sample(g_val, per_group=PER_GROUP, seed=SEED)
chosen_global_idx = val_df.iloc[chosen_idx_local].index

# guardar sólo ese subconjunto balanceado (todas las columnas)
os.makedirs(OUT_DIR, exist_ok=True)
val_balanced_path = os.path.join(OUT_DIR, "val_balanced.csv")
df.loc[chosen_global_idx].to_csv(val_balanced_path)
print(f"\n[OK] Guardado val balanceado: {val_balanced_path}  (#filas={len(chosen_global_idx)})")

# ver conteos post-selección
show_group_counts("Validación (seleccionado)", g_val[chosen_idx_local])


n_groups = 4
Validación (antes) - grupos y conteos:
  g=0: 24366
  g=1: 15759
  g=2: 1967
  g=3: 3088

[OK] Guardado val balanceado: ../datasets/jigsaw/data/dfr/val_balanced.csv  (#filas=7868)
Validación (seleccionado) - grupos y conteos:
  g=0: 1967
  g=1: 1967
  g=2: 1967
  g=3: 1967


In [8]:
if WRITE_METADATA_VAL_AS_TRAIN:
    meta_mod = df.copy()

    # Mover todo lo que era 'train' original a 'val'
    meta_mod.loc[meta_mod["split"] == "train", "split"] = "val"

    # Poner el subconjunto balanceado de validación como 'train'
    meta_mod.loc[meta_mod.index.isin(chosen_global_idx), "split"] = "train"

    new_meta_path = os.path.join(OUT_DIR, "metadata_val_as_train.csv")
    meta_mod.to_csv(new_meta_path)
    print(f"[OK] Guardado metadata para fine-tune: {new_meta_path}")


[OK] Guardado metadata para fine-tune: ../datasets/jigsaw/data/dfr/metadata_val_as_train.csv


In [7]:
import pandas as pd, os

csv_mod = "../datasets/jigsaw/data/dfr/val_balanced.csv"  # donde lo guardaste
dfm = pd.read_csv(csv_mod, index_col=0 if "Unnamed: 0" in pd.read_csv(csv_mod, nrows=1).columns else None)
print(dfm["split"].value_counts(dropna=False))

split
val    7868
Name: count, dtype: int64
